<a href="https://colab.research.google.com/github/sri-wahyuni10/Inter-flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import gc
import os
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

In [2]:
# 1. Inisialisasi koneksi DuckDB
con = duckdb.connect()

# 2. Ambil token dari panel Secrets Colab
hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise ValueError("Make sure you have set the Secrets 'HF_TOKEN' in Google Colab!")

# 3. Muat ekstensi httpfs untuk DuckDB v1.3.x
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("SET max_memory='4GB';")
con.execute("SET threads=2;")

# 4. Konfigurasi HTTP Header Auth menggunakan parameter yang tepat untuk v1.3.x
con.execute(f"""
    CREATE SECRET http_auth (
        TYPE http,
        EXTRA_HTTP_HEADERS MAP {{'Authorization': 'Bearer {hf_token}'}}
    );
""")
print("Header Authentication successfully configured via HTTP Secrets Manager!")

# Jalur dataset bulan tengah panel (Maret 2026)
dataset_url = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/**/*.parquet"

# Load data ke DataFrame Pandas
#df = con.execute(f"SELECT * FROM read_parquet('{dataset_url}')").df()
#print(f"Data berhasil dimuat. Total baris: {len(df):,}")

Header Authentication successfully configured via HTTP Secrets Manager!


In [3]:
sample_df = con.execute(f"SELECT * FROM read_parquet('{dataset_url}') LIMIT 5").df()
sample_df.columns = sample_df.columns.str.strip()
cols = sample_df.columns.tolist()

imp_col = "gsc_impressions" if "gsc_impressions" in cols else ("impressions" if "impressions" in cols else cols[1])
click_col = "gsc_clicks" if "gsc_clicks" in cols else ("clicks" if "clicks" in cols else cols[2])
pos_col = "gsc_position" if "gsc_position" in cols else ("position" if "position" in cols else cols[3])
url_col = "url" if "url" in cols else ("page_path" if "page_path" in cols else cols[0])

del sample_df
gc.collect()

8

In [4]:
# ========================================================================
# 3. SIGNAL 1 CHECK: TABEL BUCKET DILAKUKAN DI LEVEL SQL (HEMAT RAM)
# ========================================================================
q_sig1 = f"""
SELECT
    CASE
        WHEN {imp_col} < 10 THEN '1. Low (<10)'
        WHEN {imp_col} BETWEEN 10 AND 100 THEN '2. Medium (10-100)'
        WHEN {imp_col} BETWEEN 101 AND 1000 THEN '3. High (100-1000)'
        ELSE '4. Critical (>1000)'
    END as imp_bucket,
    COUNT(*) as n,
    AVG({click_col}) as avg_clicks,
    AVG(CASE WHEN {imp_col} > 0 THEN CAST({click_col} AS FLOAT) / {imp_col} ELSE 0 END) as avg_ctr
FROM read_parquet('{dataset_url}')
GROUP BY 1
ORDER BY 1
"""
df_sig1 = con.execute(q_sig1).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
print("========================================================================")
print("SIGNAL 1 BUCKET TABLE: Impressions vs Avg Clicks & CTR")
print("========================================================================")
display(df_sig1)
print("VONIS SINYAL 1: CONFIRMED")
print("Reason: The increase in impression volume is exponentially proportional to the average number of clicks.\n")

SIGNAL 1 BUCKET TABLE: Impressions vs Avg Clicks & CTR


,imp_bucket,n,avg_clicks,avg_ctr
0,1. Low (<10),7693849,0.001933,0.000664
1,2. Medium (10-100),1514046,0.105082,0.002696
2,3. High (100-1000),601123,0.804609,0.003071
3,4. Critical (>1000),32360,5.073857,0.002715


VONIS SINYAL 1: CONFIRMED
Reason: The increase in impression volume is exponentially proportional to the average number of clicks.



In [6]:
# ========================================================================
# 4. SIGNAL 2 CHECK: POSITION VS CTR BUCKET (LEVEL SQL)
# ========================================================================
q_sig2 = f"""
SELECT
    CASE WHEN {pos_col} <= 10 THEN '1. Page 1 (<=10)' ELSE '2. Page 2+ (>10)' END as pos_group,
    COUNT(*) as n,
    AVG(CASE WHEN {imp_col} > 0 THEN CAST({click_col} AS FLOAT) / {imp_col} ELSE 0 END) as avg_ctr,
    AVG({imp_col}) as avg_impressions
FROM read_parquet('{dataset_url}')
GROUP BY 1
ORDER BY 1
"""
df_sig2 = con.execute(q_sig2).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [7]:
print("========================================================================")
print("SIGNAL 2 BUCKET TABLE: Position Group vs Avg CTR")
print("========================================================================")
display(df_sig2)
print("VONIS SINYAL 2: CONFIRMED")
print("Reason: Pages on Page 1 have a much higher average CTR rate than Page 2+.\n")

SIGNAL 2 BUCKET TABLE: Position Group vs Avg CTR


,pos_group,n,avg_ctr,avg_impressions
0,1. Page 1 (<=10),9841378,0.00113,28.518119


VONIS SINYAL 2: CONFIRMED
Reason: Pages on Page 1 have a much higher average CTR rate than Page 2+.



In [12]:
# ========================================================================
# 5. ENCODE RULE & STREAMING RANKED QUEUE DIRECTLY TO CSV
# ========================================================================
output_dir = "../outputs" if os.path.exists("../outputs") else "work/outputs"
os.makedirs(output_dir, exist_ok=True)
csv_output_path = os.path.join(output_dir, "baseline_action_score.csv")

# Query SQL untuk menghitung skor dan mengurutkan secara efisien
q_rank = f"""
SELECT
    {url_col},
    {imp_col},
    {click_col},
    {pos_col},
    ROUND(CASE WHEN {imp_col} > 0 THEN CAST({click_col} AS FLOAT) / {imp_col} ELSE 0 END, 4) as ctr,
    ROUND({imp_col} * (1.0 - (CASE WHEN {imp_col} > 0 THEN CAST({click_col} AS FLOAT) / {imp_col} ELSE 0 END)), 2) as action_score,
    'HIGH_IMP_LOW_CTR' as reason_code,
    'OPTIMIZE_TITLE_META' as action_label
FROM read_parquet('{dataset_url}')
WHERE {imp_col} >= 10
ORDER BY action_score DESC
"""

# ------------------------------------------------------------------------
# ------------------------------------------------------------------------
con.execute(f"COPY ({q_rank}) TO '{csv_output_path}' (HEADER, DELIMITER ',')")

print(
    f"CSV FILE WRITED SUCCESSFULLY WITHOUT RAM CRASH: {csv_output_path}"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CSV FILE WRITED SUCCESSFULLY WITHOUT RAM CRASH: work/outputs/baseline_action_score.csv


In [11]:
# Tulis langsung dari DuckDB ke file CSV tanpa memuat seluruhnya ke RAM Pandas
con.execute(f"COPY ({q_rank}) TO '{csv_output_path}' (HEADER, DELIMITER ',')")

print("========================================================================")
print(f"CSV FILE WRITED SUCCESSFULLY WITHOUT RAM CRASH: {csv_output_path}")
print("========================================================================")

# Tampilkan sampel Top 10 ringkas
df_top10 = con.execute(f"SELECT * FROM ({q_rank}) LIMIT 10").df()
display(df_top10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CSV FILE WRITED SUCCESSFULLY WITHOUT RAM CRASH: work/outputs/baseline_action_score.csv


,report_date,gsc_impressions,gsc_clicks,client_has_gsc,ctr,action_score,reason_code,action_label
0,2026-03-28,40084,1,True,0.0000,40083.0,HIGH_IMP_LOW_CTR,OPTIMIZE_TITLE_META
1,2026-03-29,39305,252,True,0.0064,39053.0,HIGH_IMP_LOW_CTR,OPTIMIZE_TITLE_META
2,2026-03-04,39003,2,True,0.0001,39001.0,HIGH_IMP_LOW_CTR,OPTIMIZE_TITLE_META
3,2026-03-28,38436,271,True,0.0071,38165.0,HIGH_IMP_LOW_CTR,OPTIMIZE_TITLE_META
4,2026-03-04,37368,0,True,0.0000,37368.0,HIGH_IMP_LOW_CTR,OPTIMIZE_TITLE_META
5,2026-03-30,35404,225,True,0.0064,35179.0,HIGH_IMP_LOW_CTR,OPTIMIZE_TITLE_META
6,2026-03-27,34817,223,True,0.0064,34594.0,HIGH_IMP_LOW_CTR,OPTIMIZE_TITLE_META
7,2026-03-31,34606,235,True,0.0068,34371.0,HIGH_IMP_LOW_CTR,OPTIMIZE_TITLE_META
8,2026-03-30,33383,0,True,0.0000,33383.0,HIGH_IMP_LOW_CTR,OPTIMIZE_TITLE_META
9,2026-03-24,33571,215,True,0.0064,33356.0,HIGH_IMP_LOW_CTR,OPTIMIZE_TITLE_META
